# ManiSkill Mechanical Arm Rendering on Colab

This notebook is for rendering the simulated Panda arm in ManiSkill/SAPIEN and exporting MP4 videos. Use a GPU runtime first: **Runtime > Change runtime type > GPU**.


## 1. Check GPU and Vulkan

SAPIEN rendering needs Vulkan. If this section cannot find an NVIDIA GPU or Vulkan driver, restart Colab with a GPU runtime or reconnect to a new runtime.


In [ ]:
!nvidia-smi
!apt-get update -qq
!apt-get install -y -qq vulkan-tools libvulkan1 ffmpeg
!ls /usr/share/vulkan/icd.d || true
!vulkaninfo --summary || true


## 2. Pull or update the project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive
!mkdir -p Robotics_Final
%cd Robotics_Final

import os
repo = "Fanal_project_of_Robotics"
if not os.path.exists(repo):
    !git clone https://github.com/Leo0902110/Fanal_project_of_Robotics.git
else:
    %cd Fanal_project_of_Robotics
    !git pull
    %cd ..

%cd Fanal_project_of_Robotics


## 3. Install dependencies

`requirements-A.txt` includes ManiSkill/SAPIEN, Open3D, and `pin` for Panda end-effector control.


In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -r requirements-A.txt


## 4. Select the Vulkan ICD

Colab GPU runtimes normally expose an NVIDIA Vulkan ICD. This cell sets `VK_ICD_FILENAMES` when it can find one.


In [ ]:
import glob, os

icds = glob.glob('/usr/share/vulkan/icd.d/*.json')
print('ICD files:', icds)
nvidia_icds = [p for p in icds if 'nvidia' in p.lower()]
if nvidia_icds:
    os.environ['VK_ICD_FILENAMES'] = nvidia_icds[0]
    print('Using VK_ICD_FILENAMES =', os.environ['VK_ICD_FILENAMES'])
else:
    print('No NVIDIA ICD found. Continuing without VK_ICD_FILENAMES.')


## 5. Smoke tests

Run state first, then RGBD. The RGBD smoke test checks whether camera observations work before video rendering.


In [ ]:
!python main.py --mode smoke --obs-mode state --max-steps 30 --no-video --output-dir results/colab_smoke_state
!python main.py --mode smoke --obs-mode rgbd --policy scripted --max-steps 30 --no-video --output-dir results/colab_smoke_rgbd


## 6. Render active-probe mechanical arm video

Important: do **not** add `--no-video` here. The runner uses `render_mode='rgb_array'` and saves an MP4 into the output directory.


In [ ]:
!python main.py \
  --mode mvp \
  --scene pseudo_blur \
  --policy scripted \
  --use-active-probe \
  --obs-mode rgbd \
  --max-steps 120 \
  --seed 42 \
  --output-dir results/colab_render_active_probe


## 7. Display the rendered video


In [ ]:
from pathlib import Path
from IPython.display import Video, display

out_dir = Path('results/colab_render_active_probe')
print('\n'.join(str(p) for p in sorted(out_dir.glob('*'))))

videos = sorted(out_dir.glob('*.mp4'))
if videos:
    display(Video(str(videos[0]), embed=True))
else:
    print('No MP4 found. Check whether render_mode failed or --no-video was accidentally used.')


## 8. Render the active-perception decision trace

This creates a second MP4 for the ambiguity/probe/refinement decision process.


In [ ]:
!python scripts/render_decision_trace.py \
  --input results/colab_render_active_probe/active_probe_decision_trace.csv \
  --output results/active_probe_decision_trace.mp4

from IPython.display import Video, display
display(Video('results/active_probe_decision_trace.mp4', embed=True))


## 9. Fallback if rendering fails

If Vulkan/RGBD rendering fails, the simulation may still work. Use this command for metrics only, then render on a different Colab runtime or your Mac M4 local setup.


In [ ]:
!python main.py --mode mvp --scene pseudo_blur --policy scripted --use-active-probe --obs-mode state --max-steps 120 --seed 42 --no-video --output-dir results/colab_state_fallback
